# 语雀 (Yuque) API 全功能验证

> **目标**: 端到端验证语雀内部 Web API 的所有功能
> **涵盖**: 知识库、文档 CRUD、Markdown/Lake 格式、公式、表格、代码块、图片
> **认证**: Cookie 认证（_yuque_session + _ctoken）


In [12]:
from yuque_client import YuqueClient
from lake_builder import markdown_to_lake
from datetime import datetime
from pathlib import Path
from PIL import Image, ImageDraw
import json

# 初始化客户端（自动从 .env 读取 Cookie）
client = YuqueClient()
print(f"[OK] YuqueClient initialized at {datetime.now().strftime('%H:%M:%S')}")

# 获取第一个知识库用于测试
books = client.list_books()
if not books:
    raise RuntimeError("No books found. Please create a yuque repo first.")
test_book_id = books[0]["id"]
print(f"[OK] Using book: {books[0]['name']} (ID: {test_book_id})")


[OK] YuqueClient initialized at 02:03:59
[OK] Using book: LLM从入门到入土 (ID: 68025057)


## 1. 知识库与目录

验证 `list_books()` 和 `get_toc()`


In [13]:
# 列出所有知识库
books = client.list_books()
print(f"Books: {len(books)}")
for b in books[:3]:
    print(f"  - {b['name']} (ID: {b['id']}, Slug: {b['slug']})")

# 获取第一个知识库的目录
toc = client.get_toc(test_book_id)
print(f"\nTOC items: {len(toc)}")
for item in toc[:5]:
    icon = "📄" if item['type'] == 'DOC' else '📁'
    print(f"  {icon} {item['title']}")


Books: 4
  - LLM从入门到入土 (ID: 68025057, Slug: qah8x7)
  - Awesome-CS336-NoteForEveryone (ID: 68016047, Slug: zf1hbk)
  - LLM知识全景 (ID: 67989174, Slug: tp8id6)

TOC items: 238
  📄 【已更新】图片测试 - 02:00:16
  📄 【已更新】图片测试 - 01:59:48
  📄 【已更新】图片测试 - 01:58:07
  📄 【已更新】图片测试 - 01:52:21
  📄 致读者


## 2. Markdown 到 Lake HTML 转换

验证 `markdown_to_lake()` 支持的元素：
- 标题、段落、列表、引用
- 表格
- 代码块
- 数学公式（注意：\frac 必须写成 \\frac）
- 图片引用


In [14]:
# 综合测试内容（注意 LaTeX 反斜杠要双写！）
test_md = r"""
# 一级标题

## 二级标题

这是一段普通正文，包含 **加粗**、*斜体* 和 `行内代码`。

## 列表

- 无序列表项 A
- 无序列表项 B

1. 有序列表项 1
2. 有序列表项 2

## 引用

> 这是一段引用文字

## 表格

| 特性 | 支持状态 | 备注 |
|------|----------|------|
| 公式 | 支持 | Data-Latex 模式 |
| 表格 | 支持 | Lake-Table 格式 |
| 代码块 | 支持 | Card 格式 |

## 代码块

```python
def hello():
    return "Hello Yuque!"
```

## 公式

行内公式: $E = mc^2$

块级公式:
$$x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$

## 图片

![测试图片](https://cdn.nlark.com/yuque/0/2026/png/42982692/1776706558227-9cd94b0e-2e2c-4559-81e7-983400c91219.png)
"""

lake_html = markdown_to_lake(test_md)
print(f"[OK] Markdown converted to Lake HTML ({len(lake_html)} chars)")
print("\n--- Preview (first 800 chars) ---")
print(lake_html[:800])
print("\n...")
print(f"\n--- Total length: {len(lake_html)} chars ---")


[OK] Markdown converted to Lake HTML (4595 chars)

--- Preview (first 800 chars) ---
<!doctype lake><meta name="doc-version" content="1" /><meta name="viewport" content="adapt" /><meta name="typography" content="classic" /><meta name="paragraphSpacing" content="relax" />
<h1 data-lake-id="SnaUhK1T" id="SnaUhK1T"><span data-lake-id="u9rm1Fiyx" id="uWBZQ7yEp">一级标题</span></h1>
<h2 data-lake-id="EJI1OYrT" id="EJI1OYrT"><span data-lake-id="ubmeRFdtq" id="uiEx2VjZm">二级标题</span></h2>
<p data-lake-id="wPgVpX9S" id="wPgVpX9S">这是一段普通正文，包含 <span data-lake-id="uBG7CsKDR" id="uC9y8HXLi" style="font-weight: bold">加粗</span>、<span data-lake-id="uoqZBfRXx" id="uwekL9eSj" style="font-style: italic">斜体</span> 和 <code>行内代码</code>。</p>
<h2 data-lake-id="KNqNgv9R" id="KNqNgv9R"><span data-lake-id="uFGVlYqf6" id="up7G1a0yl">列表</span></h2>
<ul data-lake-id="HkqmLxeD" id="HkqmLxeD">
  <li data-lak

...

--- Total length: 4595 chars ---


## 3. 创建纯文本文档（Markdown 格式）


In [15]:
doc_plain = client.create_doc(
    book_id=test_book_id,
    title=f"纯文本测试 - {datetime.now().strftime('%H:%M:%S')}",
    content="# 纯文本测试\n\n这是一篇简单的测试文档，验证基本的 Markdown 渲染。\n\n- 列表项 1\n- 列表项 2\n\n**加粗文字** 和 *斜体文字*。",
    format="markdown"
)
print(f"[OK] Created: {doc_plain['title']}")
print(f"    Slug: {doc_plain['slug']}")
print(f"    URL: https://www.yuque.com/{test_book_id}/{doc_plain['slug']}")


[OK] Created: 纯文本测试 - 02:04:00
    Slug: ixvmfags683tn2dp
    URL: https://www.yuque.com/68025057/ixvmfags683tn2dp


## 4. 创建带公式的文档

**⚠️ 关键**: Python 字符串中 `\f` 会被当作 form feed 吃掉！
LaTeX 命令如 `\frac`、`\pm`、`\sqrt` 必须写成双反斜杠或使用原始字符串 `r"..."`


In [16]:
doc_formula = client.create_doc(
    book_id=test_book_id,
    title=f"公式测试 - {datetime.now().strftime('%H:%M:%S')}",
    content=r"""
# 数学公式验证

## 二次方程求根公式

$$x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$

## 质能方程

行内: $E = mc^2$

## 积分

$$\int_0^\infty e^{-x^2} dx = \frac{\sqrt{\pi}}{2}$$
""",
    format="markdown"
)
print(f"[OK] Created formula doc: {doc_formula['title']}")
print(f"    Slug: {doc_formula['slug']}")


[OK] Created formula doc: 公式测试 - 02:04:00
    Slug: tl4b3nwyvt8u886a


## 5. 创建带表格的文档


In [17]:
doc_table = client.create_doc(
    book_id=test_book_id,
    title=f"表格测试 - {datetime.now().strftime('%H:%M:%S')}",
    content=""""
# 表格验证

| 功能 | 状态 | 说明 |
|------|------|------|
| Markdown | 支持 | 服务端自动渲染 |
| 公式 | 支持 | LaTeX 语法 |
| 表格 | 支持 | 标准 Markdown 语法 |
| 代码块 | 支持 | Fenced code block |
| 图片 | 支持 | CDN 引用 |
""",
    format="markdown"
)
print(f"[OK] Created table doc: {doc_table['title']}")
print(f"    Slug: {doc_table['slug']}")


[OK] Created table doc: 表格测试 - 02:04:01
    Slug: yhtyuoctipgx4xkd


## 6. 创建带代码块的文档


In [18]:
doc_code = client.create_doc(
    book_id=test_book_id,
    title=f"代码块测试 - {datetime.now().strftime('%H:%M:%S')}",
    content='''
# 代码高亮验证

```python
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

print([fibonacci(i) for i in range(10)])
```

```javascript
const greeting = (name) => {
    return `Hello, ${name}!`;
};
console.log(greeting("Yuque"));
```
'''
,
    format="markdown"
)
print(f"[OK] Created code doc: {doc_code['title']}")
print(f"    Slug: {doc_code['slug']}")


[OK] Created code doc: 代码块测试 - 02:04:02
    Slug: sruwgytsf6bs6ef5


## 7. 图片上传验证

上传图片到语雀 CDN，获取公网可访问的 URL。


In [19]:
# 创建测试图片
test_img = Path('test_upload_image.png')
if not test_img.exists():
    img = Image.new('RGB', (400, 200), color='#4A90D9')
    draw = ImageDraw.Draw(img)
    draw.text((120, 90), 'Yuque Image Upload Test', fill='white')
    img.save(test_img)
    print(f"[OK] Created test image: {test_img.absolute()}")

# 上传图片
upload_result = client.upload_image(str(test_img))
print(f"[OK] Upload successful!")
print(f"    URL: {upload_result['url']}")
print(f"    filekey: {upload_result['filekey']}")

# 验证 URL 可访问
import requests
r = requests.head(upload_result['url'], timeout=10)
print(f"[OK] URL accessible: {r.status_code == 200}")


[OK] Upload successful!
    URL: https://cdn.nlark.com/yuque/0/2026/png/42982692/1776708241848-0c5bad08-5b62-47bc-939b-fc5c6cf962cd.png
    filekey: yuque/0/2026/png/42982692/1776708241848-0c5bad08-5b62-47bc-939b-fc5c6cf962cd.png
[OK] URL accessible: True


## 8. 创建带图片的文档


In [20]:
image_url = upload_result['url']

doc_image = client.create_doc(
    book_id=test_book_id,
    title=f"图片测试 - {datetime.now().strftime('%H:%M:%S')}",
    content=f"""
# 图片上传验证文档

这是一篇包含图片的测试文档。

## 上传的图片

![测试图片]({image_url})

> 图1: 语雀图片上传测试

## 说明

- 图片通过 `yuque_client.upload_image()` 上传到语雀 CDN
- URL 格式: `https://cdn.nlark.com/yuque/0/...`
- 支持在 Markdown 中直接引用
""",
    format="markdown"
)
print(f"[OK] Created image doc: {doc_image['title']}")
print(f"    Slug: {doc_image['slug']}")
print(f"    URL: https://www.yuque.com/{test_book_id}/{doc_image['slug']}")


[OK] Created image doc: 图片测试 - 02:04:03
    Slug: aa2kx5imaa0zg3v2
    URL: https://www.yuque.com/68025057/aa2kx5imaa0zg3v2


## 9. 读取文档验证

读取刚创建的带图片文档，确认内容正确。


In [21]:
# 读取带图片的文档
doc_read = client.read_doc(test_book_id, doc_image['slug'])
print(f"[OK] Read doc: {doc_read['title']}")
print(f"    ID: {doc_read['id']}")
print(f"    Format: {doc_read.get('format', 'N/A')}")

# 显示内容前 500 字符
content = doc_read.get('content', '')
print(f"\n--- Content preview ({len(content)} chars) ---")
print(content[:500])
print("\n...")


[OK] Read doc: 图片测试 - 02:04:03
    ID: 266647976
    Format: lake

--- Content preview (1898 chars) ---
<!doctype lake><meta name="doc-version" content="1" /><meta name="viewport" content="fixed" /><meta name="typography" content="classic" /><meta name="paragraphSpacing" content="relax" /><h1 data-lake-id="b2m1r" id="b2m1r"><span data-lake-id="u2cac3c79" id="u2cac3c79">图片上传验证文档</span></h1><p data-lake-id="uab4621f5" id="uab4621f5"><span data-lake-id="u004cfd65" id="u004cfd65">这是一篇包含图片的测试文档。</span></p><h2 data-lake-id="PMidh" id="PMidh"><span data-lake-id="u636aca47" id="u636aca47">上传的图片</span></h2

...


## 10. 更新文档


In [22]:
# 使用 doc_id 更新文档（不是 slug！）
doc_id = doc_read['id']

# ⚠️ 重要: update_doc 是【全量替换】，不是追加！
# 如果只传 title 不传 content → 只改标题，保留内容 ✅
# 如果传了 content → 整个文档内容被替换

# 方式1: 只更新标题（推荐，保留所有内容）
client.update_doc(
    doc_id=doc_id,
    title=f"【已更新】{doc_image['title']}",
    format="markdown"
)
print("[OK] Updated title only, content preserved")

# 方式2: 局部替换一句话（使用 replace_text）
# 后端自动读取当前内容 → 替换指定文本 → 提交
client.update_doc(
    doc_id=doc_id,
    replace_text={
        "old": "这是一篇包含图片的测试文档",
        "new": "这是一篇包含图片的【已更新】测试文档"
    },
    format="markdown"
)
print("[OK] Replaced specific sentence")


[OK] Updated title only, content preserved


ValueError: 替换失败：文档中未找到 "这是一篇包含图片的测试文档"

## 11. 清理测试文档

删除本次创建的所有测试文档（可选）。


In [ ]:
# # 收集所有测试文档的 doc_id 并删除
# # 注意：doc_plain, doc_formula, doc_table, doc_code, doc_image 的 doc_id 需要通过 read_doc 获取

# # 读取并删除
# test_docs = [
#     ("纯文本", doc_plain['slug']),
#     ("公式", doc_formula['slug']),
#     ("表格", doc_table['slug']),
#     ("代码块", doc_code['slug']),
#     ("图片", doc_image['slug']),
# ]

# for name, slug in test_docs:
#     try:
#         d = client.read_doc(test_book_id, slug)
#         client.delete_doc(d['id'], test_book_id)
#         print(f"[OK] Deleted {name} doc: {d['id']}")
#     except Exception as e:
#         print(f"[WARN] Failed to delete {name} doc: {e}")

# print("\nCleanup complete!")


## 验证总结

| 功能 | 状态 | 说明 |
|------|------|------|
| 知识库列表 | ✅ | `list_books()` |
| 目录获取 | ✅ | `get_toc()` |
| Markdown -> Lake | ✅ | `markdown_to_lake()` |
| 创建文档 | ✅ | `create_doc()` |
| 公式渲染 | ✅ | 注意 `\frac` 双反斜杠 |
| 表格渲染 | ✅ | 标准 Markdown 语法 |
| 代码高亮 | ✅ | Fenced code block |
| 图片上传 | ✅ | `upload_image()` -> CDN |
| 图片引用 | ✅ | Markdown `![alt](url)` |
| 读取文档 | ✅ | `read_doc()` |
| 更新文档 | ✅ | `update_doc()` |
| 删除文档 | ✅ | `delete_doc()` |
